# Étape 1 — Diagnostic du PSO CUDA

Ce notebook :
1. vérifie l'environnement GPU (Colab, GPU T4)
2. clone/synchronise le dépôt `DE_CUDA`
3. compile et exécute le PSO fourni par le prof
4. pose 5 questions de compréhension (section 1.6) à répondre avant de passer à l'étape 2 (DE séquentiel)

**Avant d'exécuter** : Exécution > Modifier le type d'exécution > GPU (T4).

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

## 2. Cloner / mettre à jour le dépôt

Nécessite le secret `GITHUB_TOKEN` configuré dans les Secrets Colab (tâche 0.3).

In [ ]:
from google.colab import userdata
import os

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER = "Lounismsr"
REPO = "DE_CUDA"

repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO}.git"

%cd /content
if not os.path.exists(REPO):
    !git clone {repo_url}
%cd {REPO}
!git config user.name "Ton Nom"
!git config user.email "ton.email@uha.fr"
!git pull

## 3. Compilation

In [ ]:
!nvcc -o pso src/main.cpp src/kernel.cpp src/kernel.cu

## 4. Exécution

Livrable de la tâche 1.1 : copie la sortie de cette cellule (temps GPU + minimum trouvé).

In [ ]:
!./pso

## 1.6 — Questions de compréhension [checkpoint]

Réponds à ces 5 questions (à l'écrit, ici ou dans le chat avec ton assistant) avant de passer à l'étape 2. Elles portent sur `kernel.cu` / `kernel.cpp` / `main.cpp`.

**Q1.** Le calcul du `gBest` final se fait dans une boucle sur CPU (fonction `cuda_pso` dans `kernel.cu`), après chaque appel aux kernels. Pourquoi ce choix plutôt que de calculer `gBest` directement sur GPU ? Quel est l'impact sur la performance (indice : compte le nombre et la taille des `cudaMemcpy` par itération) ?

**Q2.** Le kernel est lancé avec `threadsNum = 32` et `blocksNum = ceil(size / threadsNum)`, où `size = NUM_OF_PARTICLES * NUM_OF_DIMENSIONS`. Avec les valeurs actuelles (512 particules, 3 dimensions), combien de threads sont lancés au total ? À quoi sert le test `if (i >= NUM_OF_PARTICLES * NUM_OF_DIMENSIONS) return;` dans `kernelUpdateParticle` ?

**Q3.** Dans `kernelUpdatePBest`, il y a une condition `i % NUM_OF_DIMENSIONS != 0`. Pourquoi seul un thread par particule (et pas un thread par dimension) doit exécuter la suite de ce kernel ? Que se passerait-il sans cette condition ?

**Q4.** `tempParticle1` et `tempParticle2` sont déclarés `__device__` en mémoire globale du GPU (partagée par tous les threads), pas locaux à un thread. Est-ce correct ici (un seul thread par particule les utilise) ? Que se passerait-il si plusieurs threads par particule y écrivaient en même temps ?

**Q5.** Le protocole final demande de tester `NUM_OF_DIMENSIONS` = 10, 50, 100 et `NUM_OF_PARTICLES` = 50, 100, 500. Au-delà de changer les constantes dans `kernel.h`, quelles autres parties du code doivent changer pour que ça marche correctement (pense à `MAX_ITER`, aux bornes `START_RANGE_MIN/MAX`, et au fait que dim/pop doivent devenir des paramètres passés en ligne de commande) ?